# 04 Frozen DistilBERT Ablations

Implements manuscript Ablation C (uniform weights) and Ablation D (random weights). This notebook does not run GA optimization.

In [ ]:
from pathlib import Path
import os

# Works from the repository root, from notebooks/, or in Colab after setting BASE_DIR.
CANDIDATES = [Path.cwd(), Path.cwd().parent, Path('/content/thesis-modeling')]
BASE_DIR = next((p for p in CANDIDATES if (p / 'data' / '06_model_ready').exists()), Path('..')).resolve()
DATA_DIR = BASE_DIR / 'data' / '06_model_ready'
RESULTS_DIR = BASE_DIR / 'results'
ARTIFACTS_DIR = BASE_DIR / 'artifacts'
REPORTS_DIR = BASE_DIR / 'reports'
MODELS_DIR = BASE_DIR / 'trained_models'
for d in [RESULTS_DIR, ARTIFACTS_DIR, REPORTS_DIR, MODELS_DIR]:
    d.mkdir(parents=True, exist_ok=True)
print('BASE_DIR =', BASE_DIR)
print('DATA_DIR exists =', DATA_DIR.exists())

import json
import random
import numpy as np
import pandas as pd
from sklearn.metrics import accuracy_score, precision_recall_fscore_support, f1_score, confusion_matrix, classification_report

SEED = 42
random.seed(SEED)
np.random.seed(SEED)

LABELS = [0, 1]
TARGET_NAMES = ['ham', 'smishing']

def load_split(path):
    df = pd.read_csv(path)
    if 'model_text' not in df.columns or 'label_id' not in df.columns:
        raise ValueError(f'{path} must contain model_text and label_id')
    df['model_text'] = df['model_text'].fillna('').astype(str)
    df['label_id'] = df['label_id'].astype(int)
    return df

def binary_metrics(y_true, y_pred, model_name, split_name, seed=None):
    precision, recall, f1, _ = precision_recall_fscore_support(
        y_true, y_pred, labels=[1], average='binary', pos_label=1, zero_division=0
    )
    per_class = precision_recall_fscore_support(
        y_true, y_pred, labels=LABELS, average=None, zero_division=0
    )
    cm = confusion_matrix(y_true, y_pred, labels=LABELS)
    tn, fp, fn, tp = cm.ravel()
    row = {
        'model': model_name,
        'split': split_name,
        'seed': seed,
        'n_rows': int(len(y_true)),
        'accuracy': accuracy_score(y_true, y_pred),
        'precision_smishing': precision,
        'recall_smishing': recall,
        'f1_smishing': f1,
        'macro_f1': f1_score(y_true, y_pred, average='macro', zero_division=0),
        'weighted_f1': f1_score(y_true, y_pred, average='weighted', zero_division=0),
        'f1_ham': per_class[2][0],
        'false_negative_rate': fn / (fn + tp) if (fn + tp) else 0.0,
        'false_positive_rate': fp / (fp + tn) if (fp + tn) else 0.0,
        'tn': int(tn), 'fp': int(fp), 'fn': int(fn), 'tp': int(tp),
    }
    return row, cm

def save_classification_report(y_true, y_pred, path, title):
    text = classification_report(y_true, y_pred, labels=LABELS, target_names=TARGET_NAMES, zero_division=0)
    path.write_text(f'# {title}\n\n```text\n{text}\n```\n', encoding='utf-8')

import json
import torch
from torch import nn
from torch.utils.data import DataLoader, TensorDataset

FEATURE_DIR = ARTIFACTS_DIR / 'features'
EMBED_DIR = ARTIFACTS_DIR / 'embeddings'
ABL_C_DIR = MODELS_DIR / 'ablation_c_uniform_weights'
ABL_D_DIR = MODELS_DIR / 'ablation_d_random_weights'
for d in [ABL_C_DIR, ABL_D_DIR]:
    d.mkdir(parents=True, exist_ok=True)

EVAL_SPLITS = ['test_clean', 'test_adv_10', 'test_adv_20', 'test_adv_30']
SPLIT_PATHS = {
    'train_clean': DATA_DIR / 'clean' / 'train_clean.csv',
    'val_clean': DATA_DIR / 'clean' / 'val_clean.csv',
    'test_clean': DATA_DIR / 'clean' / 'test_clean.csv',
    'test_adv_10': DATA_DIR / 'adversarial_test' / 'test_adv_10.csv',
    'test_adv_20': DATA_DIR / 'adversarial_test' / 'test_adv_20.csv',
    'test_adv_30': DATA_DIR / 'adversarial_test' / 'test_adv_30.csv',
}

def set_torch_seed(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

def labels_for(split):
    return pd.read_csv(SPLIT_PATHS[split])['label_id'].astype(np.float32).to_numpy()

def fused(split, weights):
    emb = np.load(EMBED_DIR / f'{split}_distilbert_cls.npy').astype(np.float32)
    feats = np.load(FEATURE_DIR / f'{split}_features.npy').astype(np.float32)
    if emb.shape[0] != feats.shape[0]:
        raise ValueError(f'Row mismatch for {split}: embeddings {emb.shape}, features {feats.shape}')
    weighted = feats * weights.astype(np.float32)
    arr = np.concatenate([emb, weighted], axis=1).astype(np.float32)
    if arr.shape[1] != 776:
        raise ValueError(f'Expected 776 fused features for {split}, got {arr.shape[1]}')
    return arr

class LinearHead(nn.Module):
    def __init__(self, input_dim=776):
        super().__init__()
        self.linear = nn.Linear(input_dim, 1)
    def forward(self, x):
        return self.linear(x).squeeze(1)

def train_head(weights, seed, model_dir, max_epochs=20, patience=5, batch_size=64):
    set_torch_seed(seed)
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    x_train = fused('train_clean', weights)
    y_train = labels_for('train_clean')
    x_val = fused('val_clean', weights)
    y_val = pd.read_csv(DATA_DIR / 'clean' / 'val_clean.csv')['label_id'].astype(np.float32).to_numpy()

    train_ds = TensorDataset(torch.from_numpy(x_train), torch.from_numpy(y_train))
    train_loader = DataLoader(train_ds, batch_size=batch_size, shuffle=True, generator=torch.Generator().manual_seed(seed))
    x_val_t = torch.from_numpy(x_val).to(device)
    y_val_t = torch.from_numpy(y_val).to(device)

    model = LinearHead().to(device)
    pos = float((y_train == 1).sum())
    neg = float((y_train == 0).sum())
    pos_weight = torch.tensor([neg / pos if pos else 1.0], device=device)
    criterion = nn.BCEWithLogitsLoss(pos_weight=pos_weight)
    optimizer = torch.optim.AdamW(model.parameters(), lr=1e-3, weight_decay=0.01)

    best_state = None
    best_loss = float('inf')
    stale = 0
    history = []
    for epoch in range(1, max_epochs + 1):
        model.train()
        losses = []
        for xb, yb in train_loader:
            xb, yb = xb.to(device), yb.to(device)
            optimizer.zero_grad()
            loss = criterion(model(xb), yb)
            loss.backward()
            optimizer.step()
            losses.append(float(loss.detach().cpu()))
        model.eval()
        with torch.no_grad():
            val_loss = float(criterion(model(x_val_t), y_val_t).detach().cpu())
        history.append({'epoch': epoch, 'train_loss': float(np.mean(losses)), 'val_loss': val_loss})
        if val_loss < best_loss - 1e-5:
            best_loss = val_loss
            best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}
            stale = 0
        else:
            stale += 1
            if stale >= patience:
                break
    model.load_state_dict(best_state)
    torch.save({'state_dict': model.state_dict(), 'weights': weights, 'seed': seed, 'history': history}, model_dir / f'linear_head_seed_{seed}.pt')
    return model, device, history

@torch.no_grad()
def predict(model, device, split, weights):
    model.eval()
    x = torch.from_numpy(fused(split, weights)).to(device)
    logits = model(x).detach().cpu().numpy()
    probs = 1.0 / (1.0 + np.exp(-logits))
    return (probs >= 0.5).astype(int), probs

def run_condition(condition_name, weights, seed, model_dir):
    model, device, history = train_head(weights, seed, model_dir)
    rows = []
    pred_dir = RESULTS_DIR / 'predictions'
    pred_dir.mkdir(parents=True, exist_ok=True)
    for split in EVAL_SPLITS:
        y_true = labels_for(split).astype(int)
        y_pred, probs = predict(model, device, split, weights)
        row, cm = binary_metrics(y_true, y_pred, condition_name, split, seed=seed)
        rows.append(row)
        source = pd.read_csv(SPLIT_PATHS[split])
        pd.DataFrame({
            'final_row_id': source['final_row_id'],
            'split': split,
            'label_id': y_true,
            'pred_label_id': y_pred,
            'prob_smishing': probs,
            'model': condition_name,
            'seed': seed,
        }).to_csv(pred_dir / f'{condition_name}_{split}_seed_{seed}_predictions.csv', index=False)
    return rows, history

# Ablation C: uniform weights only.
uniform_weights = np.ones(8, dtype=np.float32)
abl_c_rows, abl_c_history = run_condition('ablation_c_uniform_weights', uniform_weights, 42, ABL_C_DIR)
abl_c_df = pd.DataFrame(abl_c_rows)
abl_c_df.to_csv(RESULTS_DIR / 'ablation_c_metrics.csv', index=False)
np.save(ABL_C_DIR / 'feature_weights.npy', uniform_weights)
(ABL_C_DIR / 'metadata.json').write_text(json.dumps({
    'model': 'Ablation C: Frozen DistilBERT with Uniform Weights',
    'weights': uniform_weights.tolist(),
    'definition': 'All eight engineered feature-group weights fixed at w_k = 1.0.',
    'not_used': 'No verification-only weighting; no GA optimization.'
}, indent=2), encoding='utf-8')
(REPORTS_DIR / 'ablation_c_summary.md').write_text(
    '# Ablation C Summary\n\n'
    'Ablation C follows the manuscript definition: frozen DistilBERT embeddings fused with eight engineered feature groups, all weighted at 1.0. '
    'It does not implement verification-only weighting. Primary metrics use smishing as positive class label_id=1.\n\n'
    + abl_c_df.to_string(index=False),
    encoding='utf-8'
)

# Ablation D: random weights only, sampled from Uniform(0, 2) for the specified seeds.
all_d_rows = []
weight_records = {}
for seed in [42, 7, 123]:
    rng = np.random.default_rng(seed)
    weights = rng.uniform(0.0, 2.0, size=8).astype(np.float32)
    np.save(ABL_D_DIR / f'feature_weights_seed_{seed}.npy', weights)
    weight_records[str(seed)] = weights.tolist()
    rows, _ = run_condition('ablation_d_random_weights', weights, seed, ABL_D_DIR)
    all_d_rows.extend(rows)
abl_d_seed_df = pd.DataFrame(all_d_rows)
metric_cols = ['accuracy', 'precision_smishing', 'recall_smishing', 'f1_smishing', 'macro_f1', 'weighted_f1', 'f1_ham', 'false_negative_rate', 'false_positive_rate']
agg = abl_d_seed_df.groupby(['model', 'split'])[metric_cols].agg(['mean', 'std']).reset_index()
agg.columns = ['_'.join([str(c) for c in col if c]) for col in agg.columns.to_flat_index()]
counts = abl_d_seed_df.groupby(['model', 'split'], as_index=False).agg(n_rows=('n_rows', 'first'), seeds=('seed', lambda s: ','.join(map(str, s))))
abl_d_metrics = counts.merge(agg, on=['model', 'split'], how='left')
abl_d_metrics.to_csv(RESULTS_DIR / 'ablation_d_metrics.csv', index=False)
abl_d_seed_df.to_csv(RESULTS_DIR / 'ablation_d_metrics_by_seed.csv', index=False)
(ABL_D_DIR / 'metadata.json').write_text(json.dumps({
    'model': 'Ablation D: Frozen DistilBERT with Random Weights',
    'weight_distribution': 'Uniform(0, 2)',
    'seeds': [42, 7, 123],
    'weights_by_seed': weight_records,
    'definition': 'Random non-optimized engineered feature-group weights held fixed per run.',
    'not_used': 'No combined veracity + rarity + difficulty weighting; no GA optimization.'
}, indent=2), encoding='utf-8')
(REPORTS_DIR / 'ablation_d_summary.md').write_text(
    '# Ablation D Summary\n\n'
    'Ablation D follows the manuscript definition: frozen DistilBERT embeddings fused with eight engineered feature groups weighted by random vectors sampled from Uniform(0, 2). '
    'Runs use seeds 42, 7, and 123 and report mean plus standard deviation. It does not implement combined veracity, rarity, or difficulty weighting. Primary metrics use smishing as positive class label_id=1.\n\n'
    '## Mean/Std Metrics\n\n' + abl_d_metrics.to_string(index=False) + '\n\n'
    '## Per-Seed Metrics\n\n' + abl_d_seed_df.to_string(index=False),
    encoding='utf-8'
)
print('Ablation C')
print(abl_c_df)
print('Ablation D aggregate')
print(abl_d_metrics)
